# Polymorphism, Duck Typing, and Operator Overloading

**Session:** 2026-11-01 · OOP with Python  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

**Polymorphism** (originating from the Greek *poly* meaning "many" and *morph* meaning "form") describes the ability of different object types to respond to the same method invocation or operator interface. 

In statically typed languages, polymorphism is often tied strictly to class inheritance hierarchies. In Python, polymorphism is fundamentally driven by **dynamic dispatch** and **Duck Typing** ("If it walks like a duck and quacks like a duck, it's a duck"). Furthermore, Python allows custom classes to integrate directly into Python's native syntax via **Operator Overloading**.

## Learning Outcomes

- Understand the distinction between Subtype Polymorphism (via inheritance) and Ad-hoc Polymorphism (via duck typing).
- Implement uniform polymorphic dispatch across heterogeneous domain models.
- Apply `functools.singledispatch` to emulate method overloading cleanly in Python's single-namespace environment.
- Implement operator overloading dunders (`__add__`, `__sub__`, `__mul__`, `__abs__`, `__eq__`, `__repr__`, `__str__`) for spatial coordinate vectors.
- Process heterogeneous geometric entities from `data/shapes_collection.json` via uniform polymorphic interfaces.

## 1. Subtype Polymorphism via Dynamic Method Dispatch

When different derived classes override a common method inherited from a base class, a client can iterate through mixed collections and invoke that method uniformly. The Python runtime automatically resolves the call to the appropriate subclass method based on the concrete object instance at runtime.

In [ ]:
class Person:
    """Base person abstraction."""

    def __init__(self, name: str) -> None:
        self.name = name

    def perform_activity(self) -> str:
        return f"{self.name} conducts daily human activities."


class Doctor(Person):
    """Medical professional subclass."""

    def perform_activity(self) -> str:
        return f"Dr. {self.name} diagnoses patients and performs clinical surgery."


class Student(Person):
    """Academic student subclass."""

    def perform_activity(self) -> str:
        return f"{self.name} attends university lectures and completes algorithmic assignments."


class Researcher(Person):
    """Scientific researcher subclass."""

    def perform_activity(self) -> str:
        return (
            f"{self.name} formulates hypotheses, executes experiments, and drafts research papers."
        )


# Heterogeneous collection treated polymorphically:
community = [
    Person(name="Alex"),
    Doctor(name="Eleanor Vance"),
    Student(name="Julian Rivera"),
    Researcher(name="Sophia Zhang"),
]

for member in community:
    print(member.perform_activity())

## 2. Duck Typing: Behavioral Protocols Without Nominal Inheritance

In Python, an object's suitability for a role is determined by the presence of specific methods and properties, rather than its inheritance pedigree:

> *"When I see a bird that walks like a duck and swims like a duck and quacks like a duck, I call that bird a duck."* — James Whitcomb Riley

Consider an audit logging system. The consumer function `execute_pipeline()` requires an object with a `log(level: str, message: str)` method. It does not care if the object inherits from a common `Logger` base class or comes from a third-party package.

In [ ]:
class ConsoleLogger:
    """Logger emitting to standard output."""

    def log(self, level: str, message: str) -> None:
        print(f"[{level.upper():5s}] (Console) {message}")


class InMemoryLogger:
    """Logger capturing events in an internal buffer."""

    def __init__(self) -> None:
        self.history: list[str] = []

    def log(self, level: str, message: str) -> None:
        entry = f"[{level.upper():5s}] (Memory) {message}"
        self.history.append(entry)


class FileLikeNullLogger:
    """Silent logger discarding messages."""

    def log(self, level: str, message: str) -> None:
        pass


def execute_pipeline(data: list[int], logger) -> float:
    """Consumes any object that quacks like a logger."""
    logger.log("info", f"Received payload of {len(data)} items.")
    if not data:
        logger.log("warn", "Payload is empty, returning 0.0.")
        return 0.0
    mean_val = sum(data) / len(data)
    logger.log("info", f"Calculated sample mean: {mean_val:.4f}")
    return mean_val


data_batch = [14, 28, 42, 56, 70]

console_log = ConsoleLogger()
memory_log = InMemoryLogger()

execute_pipeline(data_batch, console_log)
execute_pipeline(data_batch, memory_log)

print("\nMemory logger captured records:", memory_log.history)

## 3. Emulating Method Overloading in Python

In languages like C++ or Java, methods can share the same name with different parameter signatures:
```cpp
void render(int x, int y);
void render(int x, int y, Color c);
```

In Python, defining a method with the same name simply overwrites the previous definition in the class namespace dictionary (`__dict__`). To achieve method overloading, Python developers use:
1. Default argument values (`param=None`).
2. Variable-length positional/keyword arguments (`*args`, `**kwargs`).
3. Single-dispatch generic functions via `@functools.singledispatch`.

In [ ]:
from functools import singledispatchmethod


class DataFormatter:
    """Demonstrating clean method overloading using singledispatchmethod."""

    @singledispatchmethod
    def format_output(self, val) -> str:
        raise NotImplementedError(f"Cannot format unsupported type: {type(val)}")

    @format_output.register
    def _(self, val: int) -> str:
        return f"[INTEGER] {val:08d}"

    @format_output.register
    def _(self, val: float) -> str:
        return f"[FLOAT]   {val:.6f}"

    @format_output.register
    def _(self, val: str) -> str:
        return f'[STRING]  "{val.strip()}"'

    @format_output.register
    def _(self, val: list) -> str:
        return f"[LIST]    Count={len(val)} | Elements={val}"


formatter = DataFormatter()

print(formatter.format_output(42))
print(formatter.format_output(3.1415926535))
print(formatter.format_output("   Object-Oriented Programming   "))
print(formatter.format_output([10, 20, 30, 40]))

## 4. Operator Overloading with Special Dunder Methods

Python allows custom objects to intercept and respond to native operators ($+$, $-$, $*$, $==$, etc.) by implementing special double-underscore ("dunder") methods:

| Operator | Dunder Method | Operation |
| :---: | :---: | :--- |
| $\mathbf{a} + \mathbf{b}$ | `__add__(self, other)` | Addition / Vector translation |
| $\mathbf{a} - \mathbf{b}$ | `__sub__(self, other)` | Subtraction / Vector difference |
| $\mathbf{a} * \mathbf{b}$ | `__mul__(self, other)` | Scalar multiplication or dot product |
| $\|\mathbf{a}\|$ | `__abs__(self)` | Vector magnitude / Euclidean norm |
| $\mathbf{a} == \mathbf{b}$ | `__eq__(self, other)` | Equivalence relation |
| `str(a)` | `__str__(self)` | Human-readable string representation |
| `repr(a)` | `__repr__(self)` | Unambiguous programmatic representation |

Let's implement a complete mathematical 2D coordinate vector `Vector2D`.

In [ ]:
import math


class Vector2D:
    """Two-dimensional Euclidean vector with overloaded operators."""

    __slots__ = ("x", "y")

    def __init__(self, x: float, y: float) -> None:
        self.x = float(x)
        self.y = float(y)

    def __repr__(self) -> str:
        return f"Vector2D(x={self.x}, y={self.y})"

    def __str__(self) -> str:
        return f"({self.x:.2f}, {self.y:.2f})"

    def __add__(self, other: object) -> "Vector2D":
        if isinstance(other, Vector2D):
            return Vector2D(self.x + other.x, self.y + other.y)
        return NotImplemented

    def __sub__(self, other: object) -> "Vector2D":
        if isinstance(other, Vector2D):
            return Vector2D(self.x - other.x, self.y - other.y)
        return NotImplemented

    def __mul__(self, other: object) -> "Vector2D | float":
        # Scalar multiplication
        if isinstance(other, (int, float)):
            return Vector2D(self.x * other, self.y * other)
        # Dot product
        if isinstance(other, Vector2D):
            return self.x * other.x + self.y * other.y
        return NotImplemented

    def __rmul__(self, other: object) -> "Vector2D":
        # Commutative scalar multiplication: k * v
        if isinstance(other, (int, float)):
            return self.__mul__(other)
        return NotImplemented

    def __abs__(self) -> float:
        # Euclidean magnitude: sqrt(x^2 + y^2)
        return math.hypot(self.x, self.y)

    def __eq__(self, other: object) -> bool:
        if isinstance(other, Vector2D):
            return math.isclose(self.x, other.x, rel_tol=1e-9) and math.isclose(
                self.y, other.y, rel_tol=1e-9
            )
        return False


v1 = Vector2D(3.0, 4.0)
v2 = Vector2D(1.0, 2.0)

print("v1:", v1)
print("v2:", v2)
print("v1 + v2:              ", v1 + v2)
print("v1 - v2:              ", v1 - v2)
print("v1 * 2.5 (scalar):    ", v1 * 2.5)
print("3.0 * v2 (rmul):      ", 3.0 * v2)
print("v1 * v2 (dot product):", v1 * v2)
print("Magnitude |v1|:       ", abs(v1))

assert abs(v1) == 5.0
assert (v1 + v2) == Vector2D(4.0, 6.0)

## 5. Polymorphic Processing of Heterogeneous Shape Collections

Now let's load `data/shapes_collection.json` and calculate aggregate physical metrics (total surface area, total system mass, and average density) across heterogeneous shape objects.

In [ ]:
import json
from pathlib import Path

# Load shape catalog
shapes_path = Path("../data/shapes_collection.json")
if not shapes_path.exists():
    shapes_path = Path("data/shapes_collection.json")

with shapes_path.open("r", encoding="utf-8") as f:
    shape_records = json.load(f)


class ShapeEntity:
    def __init__(self, record: dict) -> None:
        self.shape_id = record["id"]
        self.stype = record["type"]
        self.color = record["color"]
        self.density = record["material_density_g_cm2"]
        self.params = record["parameters"]

    def compute_area(self) -> float:
        if self.stype == "circle":
            return math.pi * (self.params["radius"] ** 2)
        elif self.stype == "rectangle":
            return self.params["width"] * self.params["height"]
        elif self.stype == "square":
            return self.params["side"] ** 2
        elif self.stype == "triangle":
            return 0.5 * self.params["base"] * self.params["height"]
        return 0.0

    def compute_mass(self) -> float:
        return self.compute_area() * self.density


entities = [ShapeEntity(r) for r in shape_records]

total_area = sum(e.compute_area() for e in entities)
total_mass = sum(e.compute_mass() for e in entities)

print(f"Total System Components: {len(entities)}")
print(f"Total Aggregated Area:   {total_area:10.2f} cm²")
print(f"Total Aggregated Mass:   {total_mass:10.2f} grams")
print(f"Effective Mean Density:  {total_mass / total_area:10.2f} g/cm²")